# Sector-Rotation Training (Relative Targets)

Trains one LSTM per **(sector × horizon)** combination.

**Key change from previous run:** targets are now *relative* — a sector is labelled 1
if its forward return beats the median of all 8 sectors on the same day, 0 otherwise.
This gives ~50/50 class balance by construction, removing the positive market-drift
bias that inflated accuracy (and faked AUC) in the absolute-target run.

**Architecture:** same `SentimentLSTM(input=32, hidden=32, layers=2)` as per-stock.  
**Scheduler:** `ReduceLROnPlateau` — decays only when val_loss stops improving.

In [1]:
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import src
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("train_sector")

## Config

In [2]:
from src.training import ComputeConfig, TrainingConfig

CUTOFF      = "2023-06-01"
VAL_FRAC    = 0.1
PRICE_YEARS = list(range(2018, 2025))
HORIZONS    = [5, 10, 21, 42]
WINDOW      = 20
SEED        = 42

config = TrainingConfig(
    window=WINDOW,
    batch_size=16,
    n_epochs=150,
    lr=1e-3,
    weight_decay=1e-4,
    patience=20,
    scheduler="plateau",
    scheduler_patience=10,
    grad_clip=1.0,
    seed=SEED,
)

compute_config = ComputeConfig(num_workers=0)
compute_config.setup()

print(f"Device    : {compute_config.device}")
print(f"Horizons  : {HORIZONS}")
print(f"Scheduler : {config.scheduler} (patience={config.scheduler_patience})")
print(f"Epochs    : {config.n_epochs}  early-stop patience={config.patience}")

Device    : cpu
Horizons  : [5, 10, 21, 42]
Scheduler : plateau (patience=10)
Epochs    : 150  early-stop patience=20


## Load price and sentiment data

In [3]:
from src.features.sectors import SECTORS
from src.repositories.prices import PriceRepository
from src.repositories.sentiment import SentimentRepository

PRICE_DIR = Path("../data/historical-prices/prices/data/historical-prices")
SENT_DIR  = Path("../data/sentiment/data/sentiment")

price_repo = PriceRepository(data_dir=PRICE_DIR)
sent_repo  = SentimentRepository(data_dir=SENT_DIR)

all_tickers = sorted({t for tickers in SECTORS.values() for t in tickers})

price_data:     dict[str, pd.DataFrame] = {}
sentiment_data: dict[str, pd.DataFrame] = {}

for ticker in all_tickers:
    try:
        price_data[ticker]     = price_repo.load_years(ticker, PRICE_YEARS)
        sentiment_data[ticker] = sent_repo.load(ticker)
    except FileNotFoundError:
        print(f"  Missing: {ticker}")

print(f"Loaded {len(price_data)} tickers")

Loaded 48 tickers


## Pre-build sector price indices

Build each sector's equal-weight price index once and reuse it across all horizons.
This also lets us compute cross-sector relative labels before the training loop.

In [4]:
from src.features.sectors import build_sector_price_index

price_indices: dict[str, pd.DataFrame] = {}
sector_tickers: dict[str, list[str]]   = {}

for sector_name, tickers in SECTORS.items():
    available = [t for t in tickers if t in price_data]
    if len(available) < 2:
        print(f"Skipping {sector_name}: only {len(available)} tickers")
        continue
    price_indices[sector_name]  = build_sector_price_index({t: price_data[t] for t in available})
    sector_tickers[sector_name] = available
    print(f"{sector_name:<20} {len(available)} tickers  {len(price_indices[sector_name])} trading days")

Technology           10 tickers  1756 trading days
Healthcare           6 tickers  1756 trading days
Financials           6 tickers  1756 trading days
Energy               4 tickers  1756 trading days
ConsumerDisc         7 tickers  1756 trading days
ConsumerStaples      4 tickers  1756 trading days
Industrials          5 tickers  1756 trading days
UtilTelecom          6 tickers  1546 trading days


## Sanity check: class balance under relative targets

Each horizon should give ~50/50 positive rate across all sectors.
If any sector shows a strongly skewed rate, something is wrong with the label computation.

In [5]:
from src.features.sectors import compute_cross_sector_labels

print("Positive-rate sanity check (should be ~0.50 everywhere)\n")
print(f"{'Sector':<20}  " + "  ".join(f"T+{h:2d}" for h in HORIZONS))
print("-" * 60)

for sector_name in price_indices:
    rates = []
    for horizon in HORIZONS:
        labels = compute_cross_sector_labels(price_indices, horizon)
        lbl    = labels[sector_name]
        valid  = lbl[lbl >= 0]
        rates.append(f"{valid.mean():.3f}")
    print(f"{sector_name:<20}  " + "  ".join(rates))

Positive-rate sanity check (should be ~0.50 everywhere)

Sector                T+ 5  T+10  T+21  T+42
------------------------------------------------------------
Technology            0.544  0.570  0.568  0.582
Healthcare            0.493  0.517  0.531  0.517
Financials            0.509  0.531  0.549  0.604
Energy                0.446  0.429  0.416  0.398
ConsumerDisc          0.533  0.531  0.511  0.509
ConsumerStaples       0.541  0.547  0.583  0.598
Industrials           0.467  0.447  0.410  0.388
UtilTelecom           0.466  0.429  0.433  0.406


## Train: horizon × sector grid

Outer loop is **horizon** so cross-sector labels are computed once per horizon
and reused across all 8 sectors.

In [7]:
from src.features.sectors import SectorDataset, build_sector_loaders
from src.model.lstm import SentimentLSTM
from src.model.trainer import Trainer
from src.repositories.models import ModelRepository

model_repo = ModelRepository()
records: list[dict] = []

for horizon in HORIZONS:
    print(f"\n{'#'*60}")
    print(f"# HORIZON = T+{horizon}")
    print(f"{'#'*60}")

    # Compute relative labels once for all sectors at this horizon
    cross_labels = compute_cross_sector_labels(price_indices, horizon)

    for sector_name in price_indices:
        print(f"\n{'='*60}")
        print(f"{sector_name}  |  horizon=T+{horizon}")
        print(f"{'='*60}")

        available    = sector_tickers[sector_name]
        sec_prices   = {t: price_data[t]     for t in available}
        sec_sent     = {t: sentiment_data[t] for t in available}

        try:
            ds = SectorDataset(
                name=sector_name,
                price_dfs=sec_prices,
                sentiment_dfs=sec_sent,
                window=WINDOW,
                horizon=horizon,
                target_labels=cross_labels[sector_name],
            )
        except RuntimeError as exc:
            print(f"  Dataset error: {exc}")
            continue

        train_loader, val_loader, test_loader = build_sector_loaders(
            ds, cutoff=CUTOFF, val_frac=VAL_FRAC, batch_size=config.batch_size,
        )

        n_train = len(train_loader.dataset)
        n_val   = len(val_loader.dataset)
        n_test  = len(test_loader.dataset)
        pos_rate = ds.y.mean()
        print(f"  Windows — train: {n_train}, val: {n_val}, test: {n_test}  pos_rate={pos_rate:.3f}")

        if n_train == 0 or n_test == 0:
            print("  Skipping: empty split")
            continue

        model = SentimentLSTM(
            n_factors=16, sentiment_dim=768, hidden_size=32, num_layers=2, dropout=0.2,
        )
        trainer      = Trainer(model, config, compute_config)
        train_result = trainer.fit(train_loader, val_loader)

        print(
            f"  Best epoch: {train_result.best_epoch} | "
            f"val_loss: {train_result.best_val_loss:.4f} | "
            f"val_auc: {train_result.best_val_auc:.4f}"
        )

        eval_result = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)
        print(
            f"  Test AUC: {eval_result.auc_mean:.3f} "
            f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
        )
        print(
            f"  Test Acc: {eval_result.accuracy_mean:.3f} "
            f"[{eval_result.accuracy_ci_low:.3f}, {eval_result.accuracy_ci_high:.3f}]"
        )

        model_repo.save(
            f"sector_rel_{sector_name}_T{horizon}",
            model,
            {
                "sector": sector_name, "tickers": available,
                "horizon": horizon, "window": WINDOW, "mode": "relative",
                "n_train": n_train, "n_val": n_val, "n_test": n_test,
                "best_epoch": train_result.best_epoch,
                "best_val_loss": train_result.best_val_loss,
                "best_val_auc": train_result.best_val_auc,
                "test_auc": eval_result.auc_mean,
                "test_accuracy": eval_result.accuracy_mean,
                "history": train_result.history,
            },
        )

        records.append({
            "sector":      sector_name,
            "horizon":     horizon,
            "n_train":     n_train,
            "n_test":      n_test,
            "best_epoch":  train_result.best_epoch,
            "val_auc":     train_result.best_val_auc,
            "test_auc":    eval_result.auc_mean,
            "auc_ci_low":  eval_result.auc_ci_low,
            "auc_ci_high": eval_result.auc_ci_high,
            "test_acc":    eval_result.accuracy_mean,
        })

print(f"\n\nDone. {len(records)} models trained.")

15:49:43 INFO     src.features.sectors  Technology  horizon=T+5  mode=relative | 1522 windows | pos_rate=0.55 | 96% days with news
15:49:43 INFO     src.features.sectors  Technology  horizon=T+5 | train=1017  val=112  test=393



############################################################
# HORIZON = T+5
############################################################

Technology  |  horizon=T+5
  Windows — train: 1017, val: 112, test: 393  pos_rate=0.547


15:49:44 INFO     src.model.trainer  Epoch   1 | train_loss=0.8413 | val_loss=0.6635 | val_auc=0.5595 | val_acc=0.6250
15:49:44 INFO     src.model.trainer  Epoch   2 | train_loss=0.7347 | val_loss=0.6898 | val_auc=0.3245 | val_acc=0.6250
15:49:44 INFO     src.model.trainer  Epoch   3 | train_loss=0.7203 | val_loss=0.7832 | val_auc=0.4112 | val_acc=0.3750
15:49:44 INFO     src.model.trainer  Epoch   4 | train_loss=0.7161 | val_loss=0.6889 | val_auc=0.3381 | val_acc=0.6339
15:49:45 INFO     src.model.trainer  Epoch   5 | train_loss=0.6894 | val_loss=0.7221 | val_auc=0.3340 | val_acc=0.3571
15:49:46 INFO     src.model.trainer  Epoch   6 | train_loss=0.6812 | val_loss=0.7505 | val_auc=0.3371 | val_acc=0.3750
15:49:46 INFO     src.model.trainer  Epoch   7 | train_loss=0.6788 | val_loss=0.7284 | val_auc=0.3296 | val_acc=0.4286
15:49:47 INFO     src.model.trainer  Epoch   8 | train_loss=0.6666 | val_loss=0.8280 | val_auc=0.3946 | val_acc=0.3661
15:49:48 INFO     src.model.trainer  Epoch   9 |

  Best epoch: 1 | val_loss: 0.6635 | val_auc: 0.5595


15:49:58 INFO     src.features.sectors  Healthcare  horizon=T+5  mode=relative | 1522 windows | pos_rate=0.49 | 95% days with news
15:49:58 INFO     src.features.sectors  Healthcare  horizon=T+5 | train=1017  val=112  test=393


...

  Best epoch: 14 | val_loss: 0.5991 | val_auc: 0.7284


15:59:11 INFO     src.features.sectors  UtilTelecom  horizon=T+42  mode=relative | 1397 windows | pos_rate=0.39 | 82% days with news
15:59:11 INFO     src.features.sectors  UtilTelecom  horizon=T+42 | train=937  val=104  test=356


  Test AUC: 0.265 [0.216, 0.318]
  Test Acc: 0.534 [0.480, 0.587]

UtilTelecom  |  horizon=T+42
  Windows — train: 937, val: 104, test: 356  pos_rate=0.393


15:59:12 INFO     src.model.trainer  Epoch   1 | train_loss=0.7567 | val_loss=0.7969 | val_auc=0.4046 | val_acc=0.3173
15:59:12 INFO     src.model.trainer  Epoch   2 | train_loss=0.7214 | val_loss=0.5870 | val_auc=0.6175 | val_acc=0.7212
15:59:13 INFO     src.model.trainer  Epoch   3 | train_loss=0.6591 | val_loss=0.6862 | val_auc=0.4818 | val_acc=0.6154
15:59:14 INFO     src.model.trainer  Epoch   4 | train_loss=0.5654 | val_loss=0.7281 | val_auc=0.6037 | val_acc=0.5769
15:59:14 INFO     src.model.trainer  Epoch   5 | train_loss=0.4973 | val_loss=0.7997 | val_auc=0.6777 | val_acc=0.7212
15:59:15 INFO     src.model.trainer  Epoch   6 | train_loss=0.4266 | val_loss=1.0537 | val_auc=0.6124 | val_acc=0.4519
15:59:15 INFO     src.model.trainer  Epoch   7 | train_loss=0.3869 | val_loss=0.7290 | val_auc=0.6244 | val_acc=0.7500
15:59:16 INFO     src.model.trainer  Epoch   8 | train_loss=0.3517 | val_loss=0.8844 | val_auc=0.6671 | val_acc=0.6827
15:59:17 INFO     src.model.trainer  Epoch   9 |

  Best epoch: 2 | val_loss: 0.5870 | val_auc: 0.6175
  Test AUC: 0.583 [0.523, 0.642]
  Test Acc: 0.595 [0.542, 0.649]


Done. 32 models trained.


## Results

In [8]:
df = pd.DataFrame(records)

auc_pivot = df.pivot(index="sector", columns="horizon", values="test_auc")
auc_pivot.columns = [f"T+{h}" for h in auc_pivot.columns]
auc_pivot["best"] = auc_pivot.max(axis=1)
auc_pivot = auc_pivot.sort_values("best", ascending=False)

print("=== Test AUC Matrix (relative targets) ===")
print(auc_pivot.to_string(float_format="%.3f"))
print(f"\nColumn means:")
print(auc_pivot.drop(columns="best").mean().to_string(float_format="%.3f"))
print(f"\nModels with AUC > 0.55: {(df['test_auc'] > 0.55).sum()} / {len(df)}")
print(f"Models with AUC > 0.50: {(df['test_auc'] > 0.50).sum()} / {len(df)}")

=== Test AUC Matrix (relative targets) ===
                  T+5  T+10  T+21  T+42  best
sector                                       
Healthcare      0.531 0.554 0.464 0.617 0.617
Energy          0.442 0.477 0.484 0.602 0.602
UtilTelecom     0.576 0.536 0.455 0.583 0.583
Industrials     0.502 0.557 0.433 0.265 0.557
Financials      0.524 0.474 0.417 0.463 0.524
Technology      0.348 0.426 0.505 0.390 0.505
ConsumerDisc    0.470 0.445 0.490 0.503 0.503
ConsumerStaples 0.373 0.391 0.378 0.346 0.391

Column means:
T+5    0.471
T+10   0.483
T+21   0.453
T+42   0.471

Models with AUC > 0.55: 6 / 32
Models with AUC > 0.50: 12 / 32


In [9]:
# Full detail — epoch column tells us whether models actually trained
detail = df.sort_values("test_auc", ascending=False).copy()
detail["ci"] = detail.apply(
    lambda r: f"[{r['auc_ci_low']:.3f}, {r['auc_ci_high']:.3f}]", axis=1
)
print(detail[["sector", "horizon", "n_train", "n_test",
              "best_epoch", "val_auc", "test_auc", "ci", "test_acc"]]
      .to_string(index=False, float_format="%.3f"))

         sector  horizon  n_train  n_test  best_epoch  val_auc  test_auc             ci  test_acc
     Healthcare       42     1017     356           1    0.390     0.617 [0.560, 0.673]     0.570
         Energy       42     1017     356           9    0.584     0.602 [0.545, 0.662]     0.664
    UtilTelecom       42      937     356           2    0.617     0.583 [0.523, 0.642]     0.595
    UtilTelecom        5      937     393           1    0.558     0.576 [0.517, 0.631]     0.595
    Industrials       10     1017     388           1    0.604     0.557 [0.500, 0.613]     0.611
     Healthcare       10     1017     388           7    0.400     0.554 [0.498, 0.611]     0.533
    UtilTelecom       10      937     388           5    0.705     0.536 [0.472, 0.597]     0.612
     Healthcare        5     1017     393           3    0.485     0.531 [0.469, 0.591]     0.525
     Financials        5     1017     393           8    0.482     0.524 [0.464, 0.582]     0.438
     Technology     

## Ablation: Technical Features Only (No Sentiment)

Re-runs the exact same training loop but zeros out all sentiment embeddings before
building the DataLoaders. The model architecture is identical — it still has a
`sentiment_proj` layer — but it always receives a zero vector, so the LSTM input
is effectively `[tech, zeros]`.

**If the ablation AUC ≈ full-model AUC**: the FinBERT embeddings contribute nothing.
The problem is at the data / signal level, not the model level.

**If full-model AUC > ablation AUC consistently**: sentiment is adding value and
the architecture is worth refining further.

In [10]:
ablation_records: list[dict] = []

for horizon in HORIZONS:
    cross_labels = compute_cross_sector_labels(price_indices, horizon)

    for sector_name in price_indices:
        available  = sector_tickers[sector_name]
        sec_prices = {t: price_data[t]     for t in available}
        sec_sent   = {t: sentiment_data[t] for t in available}

        try:
            ds = SectorDataset(
                name=sector_name,
                price_dfs=sec_prices,
                sentiment_dfs=sec_sent,
                window=WINDOW,
                horizon=horizon,
                target_labels=cross_labels[sector_name],
            )
        except RuntimeError:
            continue

        # ── Zero out all sentiment embeddings ──────────────────────────
        import numpy as np
        ds.X_sent = np.zeros_like(ds.X_sent)
        # ───────────────────────────────────────────────────────────────

        train_loader, val_loader, test_loader = build_sector_loaders(
            ds, cutoff=CUTOFF, val_frac=VAL_FRAC, batch_size=config.batch_size,
        )
        if len(train_loader.dataset) == 0 or len(test_loader.dataset) == 0:
            continue

        model = SentimentLSTM(
            n_factors=16, sentiment_dim=768, hidden_size=32, num_layers=2, dropout=0.2,
        )
        trainer      = Trainer(model, config, compute_config)
        train_result = trainer.fit(train_loader, val_loader)
        eval_result  = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)

        print(
            f"{sector_name:<20} T+{horizon:2d} | "
            f"epoch={train_result.best_epoch:3d} | "
            f"AUC={eval_result.auc_mean:.3f} "
            f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
        )

        ablation_records.append({
            "sector":     sector_name,
            "horizon":    horizon,
            "best_epoch": train_result.best_epoch,
            "test_auc":   eval_result.auc_mean,
            "auc_ci_low": eval_result.auc_ci_low,
            "auc_ci_high":eval_result.auc_ci_high,
        })

print(f"\nDone. {len(ablation_records)} ablation models trained.")

16:35:47 INFO     src.features.sectors  Technology  horizon=T+5  mode=relative | 1522 windows | pos_rate=0.55 | 96% days with news
16:35:47 INFO     src.features.sectors  Technology  horizon=T+5 | train=1017  val=112  test=393
16:35:48 INFO     src.model.trainer  Epoch   1 | train_loss=0.8868 | val_loss=0.7289 | val_auc=0.4714 | val_acc=0.4732
16:35:48 INFO     src.model.trainer  Epoch   2 | train_loss=0.7761 | val_loss=0.6969 | val_auc=0.5388 | val_acc=0.6071
16:35:49 INFO     src.model.trainer  Epoch   3 | train_loss=0.6898 | val_loss=0.6831 | val_auc=0.5466 | val_acc=0.5804
16:35:49 INFO     src.model.trainer  Epoch   4 | train_loss=0.6997 | val_loss=0.6815 | val_auc=0.4735 | val_acc=0.5357
16:35:50 INFO     src.model.trainer  Epoch   5 | train_loss=0.6831 | val_loss=0.6651 | val_auc=0.5531 | val_acc=0.5536
16:35:50 INFO     src.model.trainer  Epoch   6 | train_loss=0.6766 | val_loss=0.6670 | val_auc=0.5418 | val_acc=0.5268
16:35:50 INFO     src.model.trainer  Epoch   7 | train_loss

Technology           T+ 5 | epoch=  5 | AUC=0.349 [0.293, 0.403]


16:36:03 INFO     src.model.trainer  Epoch   1 | train_loss=0.8373 | val_loss=0.7754 | val_auc=0.3499 | val_acc=0.4464
16:36:04 INFO     src.model.trainer  Epoch   2 | train_loss=0.7494 | val_loss=0.7999 | val_auc=0.3372 | val_acc=0.3839
16:36:04 INFO     src.model.trainer  Epoch   3 | train_loss=0.6958 | val_loss=0.8822 | val_auc=0.4131 | val_acc=0.4107
16:36:04 INFO     src.model.trainer  Epoch   4 | train_loss=0.7020 | val_loss=0.8399 | val_auc=0.3172 | val_acc=0.4107
16:36:04 INFO     src.model.trainer  Epoch   5 | train_loss=0.6781 | val_loss=0.7887 | val_auc=0.3440 | val_acc=0.3929
16:36:05 INFO     src.model.trainer  Epoch   6 | train_loss=0.6781 | val_loss=0.7890 | val_auc=0.3555 | val_acc=0.4375
16:36:06 INFO     src.model.trainer  Epoch   7 | train_loss=0.6643 | val_loss=0.8946 | val_auc=0.3633 | val_acc=0.3839
16:36:06 INFO     src.model.trainer  Epoch   8 | train_loss=0.6650 | val_loss=0.7939 | val_auc=0.3787 | val_acc=0.4286
16:36:07 INFO     src.model.trainer  Epoch   9 |

Healthcare           T+ 5 | epoch=  1 | AUC=0.505 [0.444, 0.566]


16:36:18 INFO     src.model.trainer  Epoch   1 | train_loss=0.7990 | val_loss=0.7036 | val_auc=0.5178 | val_acc=0.5000
16:36:18 INFO     src.model.trainer  Epoch   2 | train_loss=0.7751 | val_loss=0.7426 | val_auc=0.4417 | val_acc=0.4732
16:36:19 INFO     src.model.trainer  Epoch   3 | train_loss=0.7423 | val_loss=0.7071 | val_auc=0.5056 | val_acc=0.4732
16:36:20 INFO     src.model.trainer  Epoch   4 | train_loss=0.7406 | val_loss=0.7198 | val_auc=0.4876 | val_acc=0.4554
16:36:20 INFO     src.model.trainer  Epoch   5 | train_loss=0.7356 | val_loss=0.7212 | val_auc=0.4510 | val_acc=0.4732
16:36:21 INFO     src.model.trainer  Epoch   6 | train_loss=0.7058 | val_loss=0.7629 | val_auc=0.4140 | val_acc=0.4196
16:36:21 INFO     src.model.trainer  Epoch   7 | train_loss=0.7038 | val_loss=0.7039 | val_auc=0.4188 | val_acc=0.4554
16:36:21 INFO     src.model.trainer  Epoch   8 | train_loss=0.6985 | val_loss=0.7230 | val_auc=0.4240 | val_acc=0.4196
16:36:22 INFO     src.model.trainer  Epoch   9 |

...

ConsumerStaples      T+42 | epoch=  5 | AUC=0.347 [0.292, 0.403]


16:48:08 INFO     src.model.trainer  Epoch   1 | train_loss=0.7729 | val_loss=0.6082 | val_auc=0.6586 | val_acc=0.6875
16:48:08 INFO     src.model.trainer  Epoch   2 | train_loss=0.7335 | val_loss=0.6180 | val_auc=0.5562 | val_acc=0.6875
16:48:08 INFO     src.model.trainer  Epoch   3 | train_loss=0.7064 | val_loss=0.6196 | val_auc=0.6148 | val_acc=0.6250
16:48:09 INFO     src.model.trainer  Epoch   4 | train_loss=0.7065 | val_loss=0.6167 | val_auc=0.5377 | val_acc=0.6964
16:48:09 INFO     src.model.trainer  Epoch   5 | train_loss=0.6593 | val_loss=0.6158 | val_auc=0.6701 | val_acc=0.6875
16:48:10 INFO     src.model.trainer  Epoch   6 | train_loss=0.6583 | val_loss=1.0546 | val_auc=0.5117 | val_acc=0.3125
16:48:10 INFO     src.model.trainer  Epoch   7 | train_loss=0.6558 | val_loss=0.6290 | val_auc=0.5926 | val_acc=0.6696
16:48:10 INFO     src.model.trainer  Epoch   8 | train_loss=0.6484 | val_loss=0.6793 | val_auc=0.5417 | val_acc=0.6875
16:48:11 INFO     src.model.trainer  Epoch   9 |

Industrials          T+42 | epoch=  1 | AUC=0.367 [0.309, 0.428]


16:48:19 INFO     src.model.trainer  Epoch   1 | train_loss=0.8018 | val_loss=0.6839 | val_auc=0.3126 | val_acc=0.5673
16:48:19 INFO     src.model.trainer  Epoch   2 | train_loss=0.7287 | val_loss=0.8444 | val_auc=0.3444 | val_acc=0.5577
16:48:19 INFO     src.model.trainer  Epoch   3 | train_loss=0.6779 | val_loss=0.8537 | val_auc=0.2354 | val_acc=0.4231
16:48:20 INFO     src.model.trainer  Epoch   4 | train_loss=0.6582 | val_loss=0.6905 | val_auc=0.3747 | val_acc=0.5673
16:48:21 INFO     src.model.trainer  Epoch   5 | train_loss=0.6685 | val_loss=0.8006 | val_auc=0.3053 | val_acc=0.6058
16:48:21 INFO     src.model.trainer  Epoch   6 | train_loss=0.6687 | val_loss=0.6054 | val_auc=0.4120 | val_acc=0.7212
16:48:22 INFO     src.model.trainer  Epoch   7 | train_loss=0.6599 | val_loss=1.1406 | val_auc=0.2699 | val_acc=0.4808
16:48:22 INFO     src.model.trainer  Epoch   8 | train_loss=0.6502 | val_loss=0.7054 | val_auc=0.3830 | val_acc=0.6731
16:48:23 INFO     src.model.trainer  Epoch   9 |

UtilTelecom          T+42 | epoch=  6 | AUC=0.571 [0.505, 0.632]

Done. 32 ablation models trained.


In [11]:
# Side-by-side comparison: full model vs tech-only ablation
abl = pd.DataFrame(ablation_records).rename(columns={"test_auc": "auc_techonly", "best_epoch": "epoch_techonly"})
full = pd.DataFrame(records)[["sector", "horizon", "best_epoch", "test_auc"]].rename(
    columns={"test_auc": "auc_full", "best_epoch": "epoch_full"}
)
cmp = full.merge(abl[["sector", "horizon", "auc_techonly", "epoch_techonly"]], on=["sector", "horizon"])
cmp["delta"] = cmp["auc_full"] - cmp["auc_techonly"]
cmp = cmp.sort_values("delta", ascending=False)

print("=== Full model vs Tech-only (delta = full − techonly) ===\n")
print(cmp[["sector", "horizon", "epoch_full", "auc_full",
           "epoch_techonly", "auc_techonly", "delta"]]
      .to_string(index=False, float_format="%.3f"))

print(f"\nMean delta (full − techonly): {cmp['delta'].mean():+.3f}")
print(f"Cases where full > techonly : {(cmp['delta'] > 0).sum()} / {len(cmp)}")
print(f"Cases where full > techonly by >0.02: {(cmp['delta'] > 0.02).sum()} / {len(cmp)}")

=== Full model vs Tech-only (delta = full − techonly) ===

         sector  horizon  epoch_full  auc_full  epoch_techonly  auc_techonly  delta
         Energy       42           9     0.602               4         0.336  0.266
     Healthcare       42           1     0.617               4         0.373  0.244
         Energy       21           9     0.484               3         0.356  0.128
    UtilTelecom        5           1     0.576              31         0.481  0.095
     Financials       42           2     0.463               1         0.376  0.087
         Energy       10          14     0.477               6         0.414  0.063
    Industrials       10           1     0.557               1         0.510  0.047
     Healthcare       10           7     0.554               2         0.510  0.044
     Technology       21          12     0.505               6         0.477  0.028
     Financials       10           1     0.474               3         0.447  0.027
     Healthcare  

## Plan A — Scalar Sentiment Score as 17th Feature

**Hypothesis:** Instead of projecting 768-dim FinBERT embeddings (which has to compress
768 → 16 from ~1000 training windows), use `sentiment_score = 1.0·P(pos) + 0.5·P(neutral)` —
FinBERT's own directional summary already compressed into one number — appended
directly to the 16 technical features.

Changes vs the full model:
- `X_tech` shape: `(T, 17)` — column 17 is the aggregated daily sentiment score
- `X_sent` dummy: `(T, 1)` all zeros — the model's embedding path is disabled
- LSTM: `SentimentLSTM(n_factors=17, use_sentiment_proj=False)` — input size 17, no concat

**Plan B (score+delta)** follows with the 20-day rolling surprise:
`sentiment_score[t] − mean(sentiment_score[t−20:t])` appended as column 18.
Captures narrative shifts rather than the absolute sentiment level.
Both columns are min-max normalised per window by `_LazyDataset`, same as tech features.

In [ ]:
plan_a_records: list[dict] = []
plan_b_records: list[dict] = []

for sent_mode, result_list in [("score", plan_a_records), ("score+delta", plan_b_records)]:
    label = "Plan A (score)" if sent_mode == "score" else "Plan B (score+delta)"
    n_factors = 17 if sent_mode == "score" else 18
    print(f"\n{'#'*60}")
    print(f"# {label}  —  n_factors={n_factors}")
    print(f"{'#'*60}")

    for horizon in HORIZONS:
        cross_labels = compute_cross_sector_labels(price_indices, horizon)

        for sector_name in price_indices:
            available  = sector_tickers[sector_name]
            sec_prices = {t: price_data[t]     for t in available}
            sec_sent   = {t: sentiment_data[t] for t in available}

            try:
                ds = SectorDataset(
                    name=sector_name,
                    price_dfs=sec_prices,
                    sentiment_dfs=sec_sent,
                    window=WINDOW,
                    horizon=horizon,
                    target_labels=cross_labels[sector_name],
                    sentiment_mode=sent_mode,
                )
            except RuntimeError as exc:
                print(f"  {sector_name} T+{horizon}: Dataset error: {exc}")
                continue

            train_loader, val_loader, test_loader = build_sector_loaders(
                ds, cutoff=CUTOFF, val_frac=VAL_FRAC, batch_size=config.batch_size,
            )
            if len(train_loader.dataset) == 0 or len(test_loader.dataset) == 0:
                continue

            model = SentimentLSTM(
                n_factors=n_factors,
                sentiment_dim=1,          # dummy — ignored by model
                hidden_size=32,
                num_layers=2,
                dropout=0.2,
                use_sentiment_proj=False,  # tech-only LSTM, score already in X_tech
            )
            trainer      = Trainer(model, config, compute_config)
            train_result = trainer.fit(train_loader, val_loader)
            eval_result  = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)

            print(
                f"  {sector_name:<20} T+{horizon:2d} | "
                f"epoch={train_result.best_epoch:3d} | "
                f"AUC={eval_result.auc_mean:.3f} "
                f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
            )

            result_list.append({
                "sector":     sector_name,
                "horizon":    horizon,
                "best_epoch": train_result.best_epoch,
                "val_auc":    train_result.best_val_auc,
                "test_auc":   eval_result.auc_mean,
                "auc_ci_low": eval_result.auc_ci_low,
                "auc_ci_high":eval_result.auc_ci_high,
                "test_acc":   eval_result.accuracy_mean,
            })

print(f"\nPlan A: {len(plan_a_records)} models  |  Plan B: {len(plan_b_records)} models")

In [ ]:
# Four-way comparison: full embedding vs tech-only vs Plan A vs Plan B
abl_df = pd.DataFrame(ablation_records)[["sector", "horizon", "test_auc"]].rename(columns={"test_auc": "auc_techonly"})
pla_df = pd.DataFrame(plan_a_records)[["sector", "horizon", "test_auc", "best_epoch"]].rename(
    columns={"test_auc": "auc_planA", "best_epoch": "epoch_planA"})
plb_df = pd.DataFrame(plan_b_records)[["sector", "horizon", "test_auc", "best_epoch"]].rename(
    columns={"test_auc": "auc_planB", "best_epoch": "epoch_planB"})
base_df = pd.DataFrame(records)[["sector", "horizon", "best_epoch", "test_auc"]].rename(
    columns={"test_auc": "auc_emb768", "best_epoch": "epoch_emb768"})

cmp4 = base_df \
    .merge(abl_df,  on=["sector", "horizon"], how="left") \
    .merge(pla_df,  on=["sector", "horizon"], how="left") \
    .merge(plb_df,  on=["sector", "horizon"], how="left")

# Best AUC across all four approaches per row
cmp4["best_of_4"] = cmp4[["auc_emb768", "auc_techonly", "auc_planA", "auc_planB"]].max(axis=1)
cmp4["winner"]    = cmp4[["auc_emb768", "auc_techonly", "auc_planA", "auc_planB"]].idxmax(axis=1).str.replace("auc_", "")
cmp4 = cmp4.sort_values("auc_planA", ascending=False)

print("=== Four-way comparison (sorted by Plan A AUC) ===\n")
print(cmp4[["sector", "horizon", "epoch_emb768", "auc_emb768",
            "auc_techonly", "epoch_planA", "auc_planA",
            "epoch_planB", "auc_planB", "winner"]]
      .to_string(index=False, float_format="%.3f"))

print(f"\n{'─'*60}")
print(f"Mean AUC  —  emb768: {cmp4['auc_emb768'].mean():.3f}  "
      f"techonly: {cmp4['auc_techonly'].mean():.3f}  "
      f"planA: {cmp4['auc_planA'].mean():.3f}  "
      f"planB: {cmp4['auc_planB'].mean():.3f}")
print(f"Models > 0.55  —  emb768: {(cmp4['auc_emb768']>0.55).sum()}  "
      f"techonly: {(cmp4['auc_techonly']>0.55).sum()}  "
      f"planA: {(cmp4['auc_planA']>0.55).sum()}  "
      f"planB: {(cmp4['auc_planB']>0.55).sum()}")
print(f"\nWinner counts:\n{cmp4['winner'].value_counts().to_string()}")